<a href="https://colab.research.google.com/github/RATKY07/Lenguajes_programacion/blob/main/Fondo_Cripto_ITM_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 Fondo Cripto ITM
## VII Concurso de Analítica Financiera · The Trading Floor · 2026
**Instituto Tecnológico Metropolitano**

---

### 📋 Instrucciones rápidas
| Paso | Acción |
|------|---------|
| 1 | Ejecuta **Celda 1** — instala paquetes |
| 2 | Ejecuta **Celda 2** — genera `app.py` |
| 3 | Reemplaza `TU_AUTHTOKEN_AQUI` en Celda 3 con tu token de [ngrok.com](https://ngrok.com) |
| 4 | Ejecuta **Celda 3** — aparece el enlace público de la app |

---

### ✅ Ítems cubiertos de la guía
✅ Ítem 1 — Precios históricos &nbsp;|&nbsp; ✅ Ítem 2 — Rendimientos &nbsp;|&nbsp; ✅ Ítem 3 — Rendimiento acumulado

✅ Ítem 4 — Volatilidad &nbsp;|&nbsp; ✅ Ítem 5 — Maximum Drawdown &nbsp;|&nbsp; ✅ Ítem 6 — % días negativos

✅ Ítem 7 — Estrategia de trading &nbsp;|&nbsp; ✅ Ítem 8 — Backtesting vs benchmark &nbsp;|&nbsp; ✅ Ítem 9 — Análisis 24/7 + correlación

✅ **3 Portafolios:** Conservador · Moderado · Agresivo &nbsp;|&nbsp; ✅ Controles interactivos &nbsp;|&nbsp; ✅ Disclaimer IA visible

---

## Celda 1 — Instalación de dependencias

In [1]:
# =====================================================
#  CELDA 1 — INSTALACIÓN
#  VII Concurso Analítica Financiera · ITM 2026
# =====================================================
!pip install yfinance streamlit plotly pyngrok --quiet
print("\u2705 Paquetes instalados correctamente")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 61.5 MB/s eta 0:00:00
✅ Paquetes instalados correctamente


## Celda 2 — Genera el archivo `app.py`
Esta celda escribe toda la aplicación Streamlit en disco.

In [6]:
# =====================================================
#  CELDA 2 — GENERA EL ARCHIVO app.py
#  Ejecuta esta celda; luego ejecuta la Celda 3
# =====================================================

app_code = r
# -*- coding: utf-8 -*-
# =============================================================================
#  FONDO CRIPTO ITM — Aplicación Streamlit
#  VII Concurso de Analítica Financiera · The Trading Floor · 2026
#  Cumple con todos los ítems 1-9 de la guía + portafolios por perfil de riesgo
# =============================================================================

import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# ── Configuración de la página ────────────────────────────────────────────────
st.set_page_config(
    page_title="Fondo Cripto ITM",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Estilos CSS ───────────────────────────────────────────────────────────────
st.markdown(
<style>
    .main-title {
        font-size:2.4rem; font-weight:800;
        background:linear-gradient(90deg,#1a237e,#0d47a1,#1565c0);
        -webkit-background-clip:text; -webkit-text-fill-color:transparent;
        text-align:center; padding:0.5rem 0;
    }
    .subtitle { text-align:center; color:#546e7a; font-size:1.05rem; }
    .disclaimer {
        background:#fff8e1; border-left:4px solid #f9a825;
        padding:0.9rem 1.2rem; border-radius:6px;
        font-size:0.82rem; color:#5d4037; margin:1rem 0;
    }
    .portfolio-card { border-radius:12px; padding:1.4rem;
        box-shadow:0 3px 10px rgba(0,0,0,0.10); margin-bottom:1rem; }
    .cons-card { background:linear-gradient(135deg,#e8f5e9,#c8e6c9);
        border-left:5px solid #2e7d32; }
    .mod-card  { background:linear-gradient(135deg,#e3f2fd,#bbdefb);
        border-left:5px solid #1565c0; }
    .agr-card  { background:linear-gradient(135deg,#fce4ec,#f8bbd0);
        border-left:5px solid #c62828; }
    .section-header {
        font-size:1.35rem; font-weight:700; color:#1a237e;
        border-bottom:3px solid #1565c0; padding-bottom:0.4rem;
        margin:1.5rem 0 1rem 0;
    }
</style>
, unsafe_allow_html=True)


# ═════════════════════════════════════════════════════════════════════════════
# FUNCIONES AUXILIARES
# ═════════════════════════════════════════════════════════════════════════════

@st.cache_data(ttl=300)
def descargar_datos(simbolos, inicio, fin):
    Descarga precios diarios desde Yahoo Finance (datos en tiempo real).
    datos = {}
    for sym in simbolos:
        try:
            df = yf.download(sym, start=inicio, end=fin,
                             interval='1d', progress=False, auto_adjust=True)
            if not df.empty:
                if isinstance(df.columns, pd.MultiIndex):
                    df.columns = df.columns.get_level_values(0)
                datos[sym] = df[['Open','High','Low','Close','Volume']].dropna()
        except Exception as e:
            st.warning(f'No se pudo descargar {sym}: {e}')
    return datos


def calcular_retornos(precios, freq):
    # Retornos según frecuencia elegida.
    if freq == "Semanal":
        precios = precios.resample("W").last()
    elif freq == "Mensual":
        precios = precios.resample("ME").last()
    return precios.pct_change().dropna()


def calcular_metricas(retornos, nombre, simbolo, umbral_crash=5):
    # Calcula las métricas financieras de la guía (ítems 4, 5 y 6).
    n     = len(retornos)
    media = retornos.mean()
    std   = retornos.std()
    ret_acum  = (1 + retornos).prod() - 1
    vol_anual = std * np.sqrt(365)
    sharpe    = (media * 365) / (vol_anual + 1e-9)
    cum       = (1 + retornos).cumprod()
    max_dd    = (cum / cum.cummax() - 1).min()
    pct_neg   = (retornos < 0).mean() * 100
    pct_crash = (retornos < -umbral_crash / 100).mean() * 100
    var95     = np.percentile(retornos, 5)
    cvar95    = retornos[retornos <= var95].mean()
    return dict(nombre=nombre, simbolo=simbolo, n_obs=n,
                ret_medio=media, ret_acum=ret_acum,
                vol_anual=vol_anual, sharpe=sharpe,
                max_dd=max_dd, pct_neg=pct_neg,
                pct_crash=pct_crash, var95=var95, cvar95=cvar95)


def backtesting_sma(precios, sma_c, sma_l, capital):

   # ESTRATEGIA — Cruce de Medias Móviles (SMA Crossover)
   # COMPRAR: SMA rápida cruza hacia arriba la SMA lenta.
   # VENDER : SMA rápida cruza hacia abajo la SMA lenta.
   #ESPERAR: SMA rápida < SMA lenta → mantenemos efectivo.

    df = pd.DataFrame({'precio': precios.copy()})
    df["sma_c"]    = df["precio"].rolling(sma_c).mean()
    df["sma_l"]    = df["precio"].rolling(sma_l).mean()
    df["signal"]   = np.where(df["sma_c"] > df["sma_l"], 1, 0)
    df["pos"]      = df["signal"].shift(1).fillna(0)
    df["ret"]      = df["precio"].pct_change()
    df["ret_strat"]= df["pos"] * df["ret"]
    df["ret_bah"]  = df["ret"]
    df["val_strat"]= capital * (1 + df["ret_strat"]).cumprod()
    df["val_bah"]  = capital * (1 + df["ret_bah"]).cumprod()
    df["compra"]   = (df["pos"] == 1) & (df["pos"].shift(1) == 0)
    df["venta"]    = (df["pos"] == 0) & (df["pos"].shift(1) == 1)
    return df.dropna()


def construir_portafolio(datos, pesos, capital):
    # Valor histórico de un portafolio multi-activo con pesos fijos
    port = pd.DataFrame()
    for sym, w in pesos.items():
        if sym in datos and not datos[sym].empty:
            port[sym] = w * datos[sym]["Close"].pct_change().fillna(0)
    if port.empty:
        return pd.DataFrame()
    port["ret"]  = port.sum(axis=1)
    port["val"]  = capital * (1 + port["ret"]).cumprod()
    port["dd"]   = port["val"] / port["val"].cummax() - 1
    return port


# ═════════════════════════════════════════════════════════════════════════════
# CONSTANTES
# ═════════════════════════════════════════════════════════════════════════════

CRIPTO_OPT = {
    "Bitcoin (BTC)":  "BTC-USD",
    "Ethereum (ETH)": "ETH-USD",
    "Solana (SOL)":   "SOL-USD",
    "Cardano (ADA)":  "ADA-USD",
    "Litecoin (LTC)": "LTC-USD",
}
NOMBRES = {v: k.split(" (")[0] for k, v in CRIPTO_OPT.items()}
COLORES = {
    "BTC-USD":"#F7931A", "ETH-USD":"#627EEA",
    "SOL-USD":"#9945FF", "ADA-USD":"#0033AD", "LTC-USD":"#AAAAAA"
}

# ═════════════════════════════════════════════════════════════════════════════
# SIDEBAR — CONTROLES INTERACTIVOS (todos los requeridos por la guía)
# ═════════════════════════════════════════════════════════════════════════════

with st.sidebar:
    st.markdown("## 🎛️ Panel de Control")
    st.markdown("---")

    # Control 1: selección de criptomonedas
    st.markdown("### 🪙 Criptomonedas")
    sel = st.multiselect("Selecciona activos:", list(CRIPTO_OPT.keys()),
                          default=list(CRIPTO_OPT.keys()))
    simbolos = [CRIPTO_OPT[s] for s in sel]

    st.markdown("---")

    # Control 2: período de tiempo
    st.markdown("### 📅 Período")
    f_ini = st.date_input("Inicio:", value=datetime(2025, 1, 1),
                           min_value=datetime(2020, 1, 1))
    f_fin = st.date_input("Fin:", value=datetime.today(),
                           min_value=f_ini + timedelta(days=30))

    st.markdown("---")

    # Control 3: frecuencia
    st.markdown("### 📊 Frecuencia")
    freq = st.selectbox("Frecuencia:", ["Diaria", "Semanal", "Mensual"])

    st.markdown("---")

    # Control 4: monto hipotético de inversión
    st.markdown("### 💵 Inversión Hipotética")
    capital = st.number_input("Monto (USD):", 100, 1_000_000, 10_000, 500)

    st.markdown("---")

    # Parámetro opcional de estrategia (1 de 2 permitidos)
    st.markdown("⚙️ Parámetros de Estrategia")
    cripto_bt = st.selectbox("Cripto para backtesting:",
                              simbolos if simbolos else ["BTC-USD"])
    sma_c = st.slider("SMA rápida (días):", 5, 30, 7)
    sma_l = st.slider("SMA lenta  (días):", 15, 100, 21)

    st.markdown("---")

    # Umbral de pérdida extrema (elemento cripto)
    umbral = st.slider("Umbral caída extrema (%):", 1, 20, 5)


# ═════════════════════════════════════════════════════════════════════════════
# ENCABEZADO
# ═════════════════════════════════════════════════════════════════════════════

st.markdown('<h1 class="main-title">📈 Fondo Cripto ITM</h1>', unsafe_allow_html=True)
st.markdown('<p class="subtitle">VII Concurso de Analítica Financiera · The Trading Floor · 2026</p>',
            unsafe_allow_html=True)
st.markdown("""
<div class="disclaimer">
    <b>⚠️ Disclaimer IA:</b> <b>Claude Sonnet 4.6 (Anthropic)</b> — generación del código base y
    visualizaciones. El equipo definió la estrategia de trading, los portafolios, validó los cálculos
    financieros y ajustó todos los parámetros del modelo. Resultados históricos: no constituyen
    asesoría de inversión.
</div>
""", unsafe_allow_html=True)

if not simbolos:
    st.error("⚠️ Selecciona al menos una criptomoneda.")
    st.stop()

# ═════════════════════════════════════════════════════════════════════════════
# DESCARGA DE DATOS EN TIEMPO REAL
# ═════════════════════════════════════════════════════════════════════════════

with st.spinner("📡 Descargando datos en tiempo real desde Yahoo Finance…"):
    datos = descargar_datos(simbolos, str(f_ini), str(f_fin))

if not datos:
    st.error("No se pudieron descargar datos. Verifica la conexión.")
    st.stop()

# Calcular retornos y métricas para todos los activos
rets = {sym: calcular_retornos(datos[sym]["Close"], "Diaria") for sym in datos}
mets = [calcular_metricas(rets[sym], NOMBRES.get(sym, sym), sym, umbral)
        for sym in datos]


# ═════════════════════════════════════════════════════════════════════════════
# TABS
# ═════════════════════════════════════════════════════════════════════════════

t1, t2, t3, t4, t5, t6 = st.tabs([
    "📊 Análisis Base",
    "⚡ Riesgo",
    "🎯 Estrategia",
    "🌐 Cripto",
    "💼 Portafolios",
    "📋 Métricas"
])


# ─────────────────────────────────────────────────────────────────────────────
# TAB 1 — ANÁLISIS BASE (Ítems 1, 2, 3)
# ─────────────────────────────────────────────────────────────────────────────
with t1:

    # ÍTEM 1 — Precios históricos
    st.markdown('<div class="section-header">📈 Ítem 1 — Precios Históricos</div>',
                unsafe_allow_html=True)

    fig1 = go.Figure()
    for sym, df in datos.items():
        precio = df["Close"]
        if freq == "Semanal":
            precio = precio.resample("W").last()
        elif freq == "Mensual":
            precio = precio.resample("ME").last()
        pn = precio / precio.iloc[0] * 100
        fig1.add_trace(go.Scatter(x=pn.index, y=pn.values,
                                   name=NOMBRES.get(sym, sym),
                                   line=dict(color=COLORES.get(sym,"#333"), width=2.5),
                                   hovertemplate=f"<b>{NOMBRES.get(sym,sym)}</b><br>%{{x|%Y-%m-%d}}: %{{y:.1f}}<extra></extra>"))
    fig1.add_hline(y=100, line_dash="dot", line_color="gray", opacity=0.4)
    fig1.update_layout(title=f"Precios Normalizados (base 100) — Frecuencia: {freq}",
                        xaxis_title="Fecha", yaxis_title="Índice (inicio=100)",
                        template="plotly_white", height=430, hovermode="x unified",
                        legend=dict(orientation="h", y=-0.17))
    st.plotly_chart(fig1, use_container_width=True)

    # ÍTEM 2 — Rendimientos
    st.markdown('<div class="section-header">📉 Ítem 2 — Rendimientos</div>',
                unsafe_allow_html=True)

    sym_sel2 = st.selectbox("Criptomoneda:", [NOMBRES.get(s,s) for s in datos], key="sel2")
    sym2 = [s for s in datos if NOMBRES.get(s,s) == sym_sel2][0]
    ret2 = calcular_retornos(datos[sym2]["Close"], freq)

    fig2 = make_subplots(rows=2, cols=1,
                          subplot_titles=["Retornos por período (%)", "Distribución"],
                          row_heights=[0.6, 0.4], vertical_spacing=0.1)
    colores_r = ["#00c853" if r >= 0 else "#ff1744" for r in ret2.values]
    fig2.add_trace(go.Bar(x=ret2.index, y=ret2.values*100, marker_color=colores_r,
                           hovertemplate="%{x|%Y-%m-%d}: %{y:.2f}%<extra></extra>"), row=1, col=1)
    fig2.add_trace(go.Histogram(x=ret2.values*100, nbinsx=50,
                                 marker_color=COLORES.get(sym2,"#333"), opacity=0.8), row=2, col=1)
    fig2.update_layout(template="plotly_white", height=500,
                        showlegend=False, title=f"Rendimientos — {sym_sel2}")
    st.plotly_chart(fig2, use_container_width=True)

    # ÍTEM 3 — Rendimiento acumulado
    st.markdown('<div class="section-header">💰 Ítem 3 — Rendimiento Acumulado</div>',
                unsafe_allow_html=True)

    fig3 = go.Figure()
    for sym, df in datos.items():
        ret3  = calcular_retornos(df["Close"], freq)
        val3  = capital * (1 + ret3).cumprod()
        rt_t  = (val3.iloc[-1] / capital - 1) * 100
        fig3.add_trace(go.Scatter(
            x=val3.index, y=val3.values,
            name=f"{NOMBRES.get(sym,sym)} ({rt_t:+.1f}%)",
            line=dict(color=COLORES.get(sym,"#333"), width=2.5),
            hovertemplate=f"<b>{NOMBRES.get(sym,sym)}</b><br>%{{x|%Y-%m-%d}}: $%{{y:,.0f}}<extra></extra>"
        ))
    fig3.add_hline(y=capital, line_dash="dot", line_color="gray", opacity=0.5,
                    annotation_text=f"Capital inicial: ${capital:,}")
    fig3.update_layout(title=f"Rendimiento Acumulado — Inversión inicial: ${capital:,} USD",
                        xaxis_title="Fecha", yaxis_title="Valor (USD)",
                        template="plotly_white", height=420, hovermode="x unified",
                        legend=dict(orientation="h", y=-0.18))
    st.plotly_chart(fig3, use_container_width=True)

    cols3 = st.columns(len(datos))
    for i, (sym, df) in enumerate(datos.items()):
        ret_t = (1 + rets[sym]).prod() - 1
        vf    = capital * (1 + ret_t)
        cols3[i].metric(
            f"{'🟢' if ret_t>0 else '🔴'} {NOMBRES.get(sym,sym)}",
            f"${vf:,.0f}", f"{ret_t*100:+.1f}%"
        )


# ─────────────────────────────────────────────────────────────────────────────
# TAB 2 — RIESGO (Ítems 4, 5, 6)
# ─────────────────────────────────────────────────────────────────────────────
with t2:

    # ÍTEM 4 — Volatilidad
    st.markdown('<div class="section-header">📊 Ítem 4 — Volatilidad</div>',
                unsafe_allow_html=True)

    fig4a = go.Figure()
    for sym, df in datos.items():
        vol_r = rets[sym].rolling(30).std() * np.sqrt(365) * 100
        fig4a.add_trace(go.Scatter(x=vol_r.index, y=vol_r.values,
                                    name=NOMBRES.get(sym,sym),
                                    line=dict(color=COLORES.get(sym,"#333"), width=2),
                                    hovertemplate=f"<b>{NOMBRES.get(sym,sym)}</b> Vol.: %{{y:.1f}}%<extra></extra>"))
    fig4a.update_layout(title="Volatilidad Anualizada Rolling 30d (%)",
                         xaxis_title="Fecha", yaxis_title="Volatilidad (%)",
                         template="plotly_white", height=380, hovermode="x unified",
                         legend=dict(orientation="h", y=-0.18))
    st.plotly_chart(fig4a, use_container_width=True)

    col4a, col4b = st.columns(2)
    with col4a:
        nombres4 = [m["nombre"] for m in mets]
        vols4    = [m["vol_anual"]*100 for m in mets]
        cols4    = [COLORES.get(m["simbolo"],"#333") for m in mets]
        fig4b = go.Figure(go.Bar(x=nombres4, y=vols4, marker_color=cols4,
                                  text=[f"{v:.1f}%" for v in vols4], textposition="outside",
                                  hovertemplate="%{x}: %{y:.2f}%<extra></extra>"))
        fig4b.update_layout(title="Volatilidad Anual por Cripto",
                             yaxis_title="%", template="plotly_white", height=340)
        st.plotly_chart(fig4b, use_container_width=True)

    with col4b:
        # Sharpe ratio
        sharpes = [m["sharpe"] for m in mets]
        fig4c = go.Figure(go.Bar(x=nombres4, y=sharpes, marker_color=cols4,
                                  text=[f"{v:.2f}" for v in sharpes], textposition="outside",
                                  hovertemplate="%{x}: %{y:.3f}<extra></extra>"))
        fig4c.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5)
        fig4c.update_layout(title="Sharpe Ratio (mayor = mejor riesgo/retorno)",
                             yaxis_title="Sharpe", template="plotly_white", height=340)
        st.plotly_chart(fig4c, use_container_width=True)

    # ÍTEM 5 — Maximum Drawdown
    st.markdown('<div class="section-header">📉 Ítem 5 — Maximum Drawdown</div>',
                unsafe_allow_html=True)

    fig5 = go.Figure()
    for sym, df in datos.items():
        cum5 = (1 + rets[sym]).cumprod()
        dd5  = (cum5 / cum5.cummax() - 1) * 100
        fig5.add_trace(go.Scatter(x=dd5.index, y=dd5.values,
                                   name=NOMBRES.get(sym,sym),
                                   line=dict(color=COLORES.get(sym,"#333"), width=2),
                                   hovertemplate=f"<b>{NOMBRES.get(sym,sym)}</b> DD: %{{y:.2f}}%<extra></extra>"))
    fig5.update_layout(title="Maximum Drawdown Histórico (%)",
                        xaxis_title="Fecha", yaxis_title="Drawdown (%)",
                        template="plotly_white", height=390, hovermode="x unified",
                        legend=dict(orientation="h", y=-0.18))
    st.plotly_chart(fig5, use_container_width=True)

    # ÍTEM 6 — % días negativos
    st.markdown('<div class="section-header">📋 Ítem 6 — Frecuencia de Pérdidas</div>',
                unsafe_allow_html=True)

    col6a, col6b = st.columns(2)
    with col6a:
        pct_neg   = [m["pct_neg"] for m in mets]
        pct_crash = [m["pct_crash"] for m in mets]
        fig6a = go.Figure(data=[
            go.Bar(name="% días negativos", x=nombres4, y=pct_neg,
                   marker_color="#ef5350", text=[f"{v:.1f}%" for v in pct_neg], textposition="outside"),
            go.Bar(name=f"% caídas >{umbral}%", x=nombres4, y=pct_crash,
                   marker_color="#b71c1c", text=[f"{v:.1f}%" for v in pct_crash], textposition="outside"),
        ])
        fig6a.update_layout(barmode="group", title="Frecuencia de Pérdidas",
                             yaxis_title="%", template="plotly_white", height=370,
                             legend=dict(orientation="h", y=-0.22))
        st.plotly_chart(fig6a, use_container_width=True)

    with col6b:
        var_vals  = [m["var95"]*100 for m in mets]
        cvar_vals = [m["cvar95"]*100 for m in mets]
        fig6b = go.Figure(data=[
            go.Bar(name="VaR 95%",  x=nombres4, y=var_vals,
                   marker_color="#ff7043", text=[f"{v:.2f}%" for v in var_vals], textposition="outside"),
            go.Bar(name="CVaR 95%", x=nombres4, y=cvar_vals,
                   marker_color="#d32f2f", text=[f"{v:.2f}%" for v in cvar_vals], textposition="outside"),
        ])
        fig6b.update_layout(barmode="group", title="VaR y CVaR al 95%",
                             yaxis_title="Pérdida (%)", template="plotly_white", height=370,
                             legend=dict(orientation="h", y=-0.22))
        st.plotly_chart(fig6b, use_container_width=True)


# ─────────────────────────────────────────────────────────────────────────────
# TAB 3 — ESTRATEGIA & BACKTESTING (Ítems 7 y 8)
# ─────────────────────────────────────────────────────────────────────────────
with t3:

    # ÍTEM 7 — Idea de trading
    st.markdown('<div class="section-header">🎯 Ítem 7 — Idea de Trading del Fondo</div>',
                unsafe_allow_html=True)
    st.info(f"""
    **📌 Estrategia: Cruce de Medias Móviles (SMA Crossover)**

    | Señal | Condición | Acción |
    |-------|-----------|--------|
    | 🟢 COMPRAR | SMA({sma_c}d) cruza **hacia arriba** la SMA({sma_l}d) | Entrar al mercado |
    | 🔴 VENDER  | SMA({sma_c}d) cruza **hacia abajo** la SMA({sma_l}d) | Salir → mantener efectivo |
    | ⬜ ESPERAR | SMA corta < SMA lenta | Sin posición |

    **Justificación:** Las medias móviles filtran el ruido de alta frecuencia típico del mercado
    cripto 24/7. La media rápida (SMA {sma_c}d) captura momentum de corto plazo; la lenta
    (SMA {sma_l}d) confirma la tendencia. Esta regla es simple, clara y defendible ante cualquier
    inversionista.
    """)

    # ÍTEM 8 — Backtesting
    st.markdown('<div class="section-header">📈 Ítem 8 — Backtesting vs Buy & Hold</div>',
                unsafe_allow_html=True)

    if cripto_bt in datos:
        df_bt = backtesting_sma(datos[cripto_bt]["Close"], sma_c, sma_l, capital)

        fig_bt = make_subplots(rows=3, cols=1, shared_xaxes=True,
                                row_heights=[0.5, 0.33, 0.17],
                                subplot_titles=[
                                    f"Portafolio: Estrategia SMA vs Buy&Hold — {NOMBRES.get(cripto_bt,'')}",
                                    "Precio + Medias Móviles + Señales",
                                    "Posición (1=mercado, 0=efectivo)"
                                ],
                                vertical_spacing=0.06)

        # Panel 1: valor del portafolio
        fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt["val_strat"],
                                     name=f"SMA({sma_c}/{sma_l})",
                                     line=dict(color="#1565c0", width=2.5)), row=1, col=1)
        fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt["val_bah"],
                                     name="Buy & Hold",
                                     line=dict(color="#f9a825", width=2, dash="dash")), row=1, col=1)
        fig_bt.add_hline(y=capital, line_dash="dot", line_color="gray", opacity=0.3, row=1, col=1)

        # Panel 2: precio + SMAs + señales
        fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt["precio"],
                                     name="Precio", line=dict(color="#90a4ae", width=1.2)), row=2, col=1)
        fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt["sma_c"],
                                     name=f"SMA {sma_c}d", line=dict(color="#e53935", width=2)), row=2, col=1)
        fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt["sma_l"],
                                     name=f"SMA {sma_l}d", line=dict(color="#1565c0", width=2, dash="dot")), row=2, col=1)

        compras = df_bt[df_bt["compra"]]
        ventas  = df_bt[df_bt["venta"]]
        fig_bt.add_trace(go.Scatter(x=compras.index, y=compras["precio"], mode="markers",
                                     name="Compra", marker=dict(symbol="triangle-up", size=11, color="#00e676")), row=2, col=1)
        fig_bt.add_trace(go.Scatter(x=ventas.index, y=ventas["precio"], mode="markers",
                                     name="Venta", marker=dict(symbol="triangle-down", size=11, color="#ff5252")), row=2, col=1)

        # Panel 3: posición
        fig_bt.add_trace(go.Scatter(x=df_bt.index, y=df_bt["pos"], name="Posición",
                                     line=dict(color="#7c4dff", width=1.5),
                                     fill="tozeroy", fillcolor="rgba(124,77,255,0.1)"), row=3, col=1)

        fig_bt.update_layout(template="plotly_white", height=640, hovermode="x unified",
                              legend=dict(orientation="h", y=-0.08))
        fig_bt.update_yaxes(title_text="USD", row=1, col=1)
        fig_bt.update_yaxes(title_text="Precio", row=2, col=1)
        fig_bt.update_yaxes(title_text="Pos.", row=3, col=1, range=[-0.1, 1.3])
        st.plotly_chart(fig_bt, use_container_width=True)

        # Métricas de desempeño
        vf_e  = df_bt["val_strat"].iloc[-1]
        vf_b  = df_bt["val_bah"].iloc[-1]
        n_ops = len(compras) + len(ventas)
        t_mkt = df_bt["pos"].mean() * 100

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("💼 Estrategia SMA",  f"${vf_e:,.0f}", f"{(vf_e/capital-1)*100:+.1f}%")
        c2.metric("📊 Buy & Hold",      f"${vf_b:,.0f}", f"{(vf_b/capital-1)*100:+.1f}%")
        c3.metric("🔄 Operaciones",     f"{n_ops}", f"▲{len(compras)} ▼{len(ventas)}")
        c4.metric("⏱️ Tiempo en Mercado", f"{t_mkt:.1f}%", "Resto en efectivo")

        # ¿Quién ganó?
        if vf_e > vf_b:
            st.success(f"✅ La estrategia SMA supera al Buy&Hold en ${vf_e-vf_b:,.0f} USD")
        else:
            st.warning(f"⚠️ Buy&Hold supera a SMA en ${vf_b-vf_e:,.0f} USD — "
                       "La estrategia reduce riesgo al salir del mercado en caídas.")

        # Tabla comparativa de riesgo
        re_s = df_bt["ret_strat"].dropna()
        re_b = df_bt["ret_bah"].dropna()
        vol_s = re_s.std()*np.sqrt(365)*100
        vol_b = re_b.std()*np.sqrt(365)*100
        dd_s  = ((df_bt["val_strat"]/df_bt["val_strat"].cummax())-1).min()*100
        dd_b  = ((df_bt["val_bah"]  /df_bt["val_bah"].cummax()  )-1).min()*100

        df_comp = pd.DataFrame({
            "Métrica": ["Retorno Total", "Volatilidad Anual", "Máx. Drawdown", "Sharpe Aprox."],
            f"SMA({sma_c}/{sma_l})": [
                f"{(vf_e/capital-1)*100:.2f}%", f"{vol_s:.2f}%", f"{dd_s:.2f}%",
                f"{(re_s.mean()*365)/(vol_s/100+1e-9):.2f}"
            ],
            "Buy & Hold": [
                f"{(vf_b/capital-1)*100:.2f}%", f"{vol_b:.2f}%", f"{dd_b:.2f}%",
                f"{(re_b.mean()*365)/(vol_b/100+1e-9):.2f}"
            ]
        })
        st.dataframe(df_comp, use_container_width=True, hide_index=True)

    else:
        st.warning("Selecciona una criptomoneda disponible para el backtesting.")


# ─────────────────────────────────────────────────────────────────────────────
# TAB 4 — ELEMENTO PROPIO CRIPTO (Ítem 9)
# ─────────────────────────────────────────────────────────────────────────────
with t4:
    st.markdown('<div class="section-header">🌐 Ítem 9 — Elemento Propio del Mercado Cripto</div>',
                unsafe_allow_html=True)
    st.info("""
    **Elemento diferenciador: Efecto Día de la Semana + Análisis de Correlación**

    Las criptomonedas operan **365 días × 24 horas**, a diferencia de las acciones tradicionales.
    Esto genera un efecto estadístico único: el retorno promedio varía sistemáticamente según el
    día de la semana (*day-of-week effect*), inexistente en bolsas con horario fijo.
    """)

    # Efecto día de la semana
    dias = ["Lunes","Martes","Miércoles","Jueves","Viernes","Sábado","Domingo"]
    fig9a = go.Figure()
    for sym, df in datos.items():
        r_d = datos[sym]["Close"].pct_change().dropna()
        r_d.index = pd.to_datetime(r_d.index)
        d_df = r_d.to_frame("r")
        d_df["dia"] = d_df.index.dayofweek
        med_dia = d_df.groupby("dia")["r"].mean() * 100
        med_dia.index = [dias[i] for i in med_dia.index]
        fig9a.add_trace(go.Bar(x=med_dia.index, y=med_dia.values,
                                name=NOMBRES.get(sym,sym),
                                marker_color=COLORES.get(sym,"#333"), opacity=0.85,
                                hovertemplate=f"<b>{NOMBRES.get(sym,sym)}</b><br>%{{x}}: %{{y:.3f}}%<extra></extra>"))

    fig9a.add_hline(y=0, line_color="gray", line_dash="dot", opacity=0.5)
    fig9a.update_layout(
        title="Retorno Medio por Día de la Semana — Mercado 24/7",
        xaxis_title="Día", yaxis_title="Retorno medio (%)",
        barmode="group", template="plotly_white", height=410,
        legend=dict(orientation="h", y=-0.18)
    )
    st.plotly_chart(fig9a, use_container_width=True)
    st.caption("**Fines de semana:** menor volumen institucional, mayor predominancia minorista "
               "y menor liquidez → movimientos más erráticos. Este patrón no tiene equivalente "
               "en mercados de renta variable.")

    # Correlación
    st.markdown("---")
    st.subheader("🔗 Correlación entre Criptomonedas")
    st.caption("Correlación alta → baja diversificación real entre activos.")

    if len(datos) >= 2:
        ret_mx = pd.DataFrame({NOMBRES.get(s,s): rets[s] for s in datos}).dropna()
        corr   = ret_mx.corr()

        fig9b = go.Figure(go.Heatmap(
            z=corr.values, x=corr.columns.tolist(), y=corr.index.tolist(),
            colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
            text=[[f"{v:.2f}" for v in row] for row in corr.values],
            texttemplate="%{text}",
            hovertemplate="%{x} — %{y}: %{z:.3f}<extra></extra>"
        ))
        fig9b.update_layout(title="Matriz de Correlación de Retornos Diarios",
                             height=420, template="plotly_white")
        st.plotly_chart(fig9b, use_container_width=True)

    # Crashes extremos
    st.markdown("---")
    st.subheader(f"💥 Días con Caídas Extremas (> {umbral}%)")
    crash_data = []
    for sym, df in datos.items():
        r_d  = df["Close"].pct_change().dropna()
        crs  = r_d[r_d < -umbral/100]
        if not crs.empty:
            crash_data.append({
                "Criptomoneda": NOMBRES.get(sym,sym),
                "N° Caídas":    len(crs),
                "Peor caída":   f"{crs.min()*100:.2f}%",
                "Promedio":     f"{crs.mean()*100:.2f}%",
                "% del período":f"{len(crs)/len(r_d)*100:.1f}%",
            })
    if crash_data:
        st.dataframe(pd.DataFrame(crash_data), use_container_width=True, hide_index=True)


# ─────────────────────────────────────────────────────────────────────────────
# TAB 5 — PORTAFOLIOS
# ─────────────────────────────────────────────────────────────────────────────
with t5:
    st.markdown('<div class="section-header">💼 Portafolios por Perfil de Riesgo</div>',
                unsafe_allow_html=True)

    # Definición de los tres portafolios
    # CONSERVADOR: 70% BTC + 20% ETH → prioriza preservación de capital
    # MODERADO:    50% BTC + 30% ETH + 10% SOL + 5% ADA + 5% LTC → balance
    # AGRESIVO:    25% BTC + 25% ETH + 30% SOL + 15% ADA + 5% LTC → max retorno

    portafolios = {
        "🛡️ Conservador": {
            "pesos":  {"BTC-USD":0.70,"ETH-USD":0.20,"LTC-USD":0.05,"ADA-USD":0.05},
            "clase":  "cons-card",
            "color":  "#2e7d32",
            "tesis":  ("**Tesis:** 70% Bitcoin (oro digital, menor volatilidad relativa) + "
                       "20% Ethereum (ecosistema maduro). Prioriza preservación de capital.\n\n"
                       "**Recomendado para:** Horizonte > 2 años, baja tolerancia a pérdidas."),
        },
        "⚖️ Moderado": {
            "pesos":  {"BTC-USD":0.50,"ETH-USD":0.30,"SOL-USD":0.10,"ADA-USD":0.05,"LTC-USD":0.05},
            "clase":  "mod-card",
            "color":  "#1565c0",
            "tesis":  ("**Tesis:** BTC como ancla (50%) + ETH (DeFi) + SOL (alta velocidad). "
                       "Diversificación en 5 activos reduce el riesgo idiosincrático.\n\n"
                       "**Recomendado para:** Horizonte 1-3 años, tolerancia media al riesgo."),
        },
        "🚀 Agresivo": {
            "pesos":  {"BTC-USD":0.25,"ETH-USD":0.25,"SOL-USD":0.30,"ADA-USD":0.15,"LTC-USD":0.05},
            "clase":  "agr-card",
            "color":  "#c62828",
            "tesis":  ("**Tesis:** Sobreponderamos SOL (30%) + ADA (15%) por su mayor potencial "
                       "de crecimiento. BTC y ETH al 25% c/u como base de liquidez.\n\n"
                       "**Recomendado para:** Horizonte > 3 años, alta tolerancia al riesgo."),
        },
    }

    # Normalizar pesos a activos disponibles
    port_calc = {}
    for nombre_p, info_p in portafolios.items():
        pf = {s: w for s, w in info_p["pesos"].items() if s in datos}
        tot = sum(pf.values())
        if tot > 0:
            port_calc[nombre_p] = {s: w/tot for s, w in pf.items()}

    # Tarjetas y gráficos de dona
    cols_p = st.columns(3)
    resultados_p = {}

    for idx, (nombre_p, info_p) in enumerate(portafolios.items()):
        pesos_n = port_calc.get(nombre_p, {})
        with cols_p[idx]:
            if not pesos_n:
                st.warning(f"Sin datos para {nombre_p}")
                continue

            df_p = construir_portafolio(datos, pesos_n, capital)
            resultados_p[nombre_p] = (info_p, pesos_n, df_p)

            if df_p.empty:
                continue

            ret_p  = (df_p["val"].iloc[-1] / capital - 1) * 100
            vol_p  = df_p["ret"].std() * np.sqrt(365) * 100
            dd_p   = df_p["dd"].min() * 100
            sh_p   = (df_p["ret"].mean()*365) / (df_p["ret"].std()*np.sqrt(365)+1e-9)

            st.markdown(f"""
            <div class="portfolio-card {info_p['clase']}">
                <h3 style="color:{info_p['color']};margin:0 0 0.7rem 0">{nombre_p}</h3>
                <div style="display:grid;grid-template-columns:1fr 1fr;gap:0.4rem;margin-bottom:0.6rem">
                    <div><b>Retorno</b><br>
                    <span style="font-size:1.5rem;font-weight:800;color:{info_p['color']}">{ret_p:+.1f}%</span></div>
                    <div><b>Valor final</b><br><span style="font-size:1.15rem;font-weight:700">${df_p['val'].iloc[-1]:,.0f}</span></div>
                    <div><b>Volatilidad</b><br>{vol_p:.1f}% anual</div>
                    <div><b>Max Drawdown</b><br>{dd_p:.1f}%</div>
                    <div colspan="2"><b>Sharpe</b><br>{sh_p:.2f}</div>
                </div>
            </div>
            """, unsafe_allow_html=True)

            # Tesis
            st.markdown(info_p["tesis"])

            # Dona de pesos
            fig_dona = go.Figure(go.Pie(
                labels=[NOMBRES.get(s,s) for s in pesos_n],
                values=list(pesos_n.values()), hole=0.55,
                marker_colors=[COLORES.get(s,"#333") for s in pesos_n],
                textinfo="label+percent",
                hovertemplate="%{label}: %{percent}<extra></extra>"
            ))
            fig_dona.update_layout(showlegend=False, height=270,
                                    margin=dict(t=20,b=10,l=10,r=10))
            st.plotly_chart(fig_dona, use_container_width=True)

    # Comparativa de evolución
    st.markdown('<div class="section-header">📊 Comparativa de los Tres Portafolios</div>',
                unsafe_allow_html=True)

    colores_p = {"🛡️ Conservador":"#2e7d32","⚖️ Moderado":"#1565c0","🚀 Agresivo":"#c62828"}
    fig_cmp = go.Figure()
    for nombre_p, (info_p, pesos_n, df_p) in resultados_p.items():
        if df_p.empty: continue
        fig_cmp.add_trace(go.Scatter(
            x=df_p.index, y=df_p["val"], name=nombre_p,
            line=dict(color=colores_p.get(nombre_p,"#333"), width=2.5),
            hovertemplate=f"<b>{nombre_p}</b><br>%{{x|%Y-%m-%d}}: $%{{y:,.0f}}<extra></extra>"
        ))
    fig_cmp.add_hline(y=capital, line_dash="dot", line_color="gray", opacity=0.4,
                       annotation_text=f"Capital inicial: ${capital:,}")
    fig_cmp.update_layout(title=f"Evolución Comparativa — Capital inicial: ${capital:,} USD",
                           xaxis_title="Fecha", yaxis_title="Valor (USD)",
                           template="plotly_white", height=420, hovermode="x unified",
                           legend=dict(orientation="h", y=-0.15))
    st.plotly_chart(fig_cmp, use_container_width=True)

    # Tabla comparativa
    tabla_p = []
    for nombre_p, (info_p, pesos_n, df_p) in resultados_p.items():
        if df_p.empty: continue
        tabla_p.append({
            "Portafolio":    nombre_p,
            "Retorno Total": f"{(df_p['val'].iloc[-1]/capital-1)*100:+.2f}%",
            "Valor Final":   f"${df_p['val'].iloc[-1]:,.0f}",
            "Volatilidad":   f"{df_p['ret'].std()*np.sqrt(365)*100:.2f}%",
            "Max Drawdown":  f"{df_p['dd'].min()*100:.2f}%",
            "Sharpe":        f"{(df_p['ret'].mean()*365)/(df_p['ret'].std()*np.sqrt(365)+1e-9):.3f}",
        })
    if tabla_p:
        st.dataframe(pd.DataFrame(tabla_p), use_container_width=True, hide_index=True)


# ─────────────────────────────────────────────────────────────────────────────
# TAB 6 — MÉTRICAS COMPLETAS
# ─────────────────────────────────────────────────────────────────────────────
with t6:
    st.markdown('<div class="section-header">📋 Tabla de Métricas por Criptomoneda</div>',
                unsafe_allow_html=True)

    df_tabla = pd.DataFrame([{
        "Criptomoneda":        m["nombre"],
        "Retorno Acumulado":   f"{m['ret_acum']*100:+.2f}%",
        "Ret. Medio Diario":   f"{m['ret_medio']*100:+.3f}%",
        "Volatilidad Anual":   f"{m['vol_anual']*100:.2f}%",
        "Sharpe Ratio":        f"{m['sharpe']:.3f}",
        "Máx. Drawdown":       f"{m['max_dd']*100:.2f}%",
        "% Días Negativos":    f"{m['pct_neg']:.1f}%",
        f"% Caídas>{umbral}%": f"{m['pct_crash']:.1f}%",
        "VaR 95%":             f"{m['var95']*100:.2f}%",
        "CVaR 95%":            f"{m['cvar95']*100:.2f}%",
    } for m in mets])

    st.dataframe(df_tabla, use_container_width=True, hide_index=True)

    # Radar chart comparativo
    st.markdown("---")
    st.subheader("🕸️ Perfil Multi-Métrica por Criptomoneda")
    st.caption("Mayor área = mejor perfil global de riesgo/retorno")

    cats = ["Retorno Acum.", "Vol. (inv.)", "Sharpe", "DD (inv.)", "% Días +"]
    fig_r = go.Figure()
    for m in mets:
        if m["simbolo"] not in datos: continue
        vals = [
            min(max((m["ret_acum"]+1)*50, 0), 100),
            min(max(100 - m["vol_anual"]*100, 0), 100),
            min(max((m["sharpe"]+2)*25, 0), 100),
            min(max(100 + m["max_dd"]*100, 0), 100),
            min(max(100 - m["pct_neg"], 0), 100),
        ]
        fig_r.add_trace(go.Scatterpolar(
            r=vals+[vals[0]], theta=cats+[cats[0]],
            name=m["nombre"],
            line=dict(color=COLORES.get(m["simbolo"],"#333"), width=2),
            fill="toself", fillcolor=COLORES.get(m["simbolo"],"#333"),
            opacity=0.18
        ))
    fig_r.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0,100])),
        template="plotly_white", height=460,
        legend=dict(orientation="h", y=-0.12)
    )
    st.plotly_chart(fig_r, use_container_width=True)
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ app.py escrito correctamente")
print(f"📏 Líneas de código: {app_code.count(chr(10)):,}")

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

print("\u2705 app.py generado correctamente")
print(f"\U0001f4cf Lineas: {app_code.count(chr(10)):,}")


SyntaxError: invalid decimal literal (3696909857.py, line 37)

## Celda 3 — Lanza la app
⚠️ **Antes de ejecutar:** reemplaza `TU_AUTHTOKEN_AQUI` con tu token de [ngrok.com](https://ngrok.com) (registro gratuito, tarda 1 minuto).

Si no quieres crear cuenta, descomenta el bloque de **localtunnel** al final de la celda.

In [4]:
# =====================================================
#  CELDA 3 — LANZAR LA APLICACIÓN
#  Reemplaza TU_AUTHTOKEN_AQUI con tu token de:
#  https://ngrok.com  (registro gratuito)
# =====================================================

import subprocess, time
from pyngrok import ngrok

NGROK_TOKEN = "3DjAmabQ3IsEayYU1f03CIfrWaq_3UtHjHjQSbUkfEyU9sEFS"  # <- REEMPLAZA AQUÍ

ngrok.set_auth_token(NGROK_TOKEN)
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false"
])
time.sleep(6)

tunnel = ngrok.connect(8501, "http")
print("=" * 60)
print("\U0001f680  APP EN VIVO:")
print(f"    {tunnel.public_url}")
print("=" * 60)
print("Comparte este enlace en el stand.")
print("Para detener: ngrok.kill()")

# ── ALTERNATIVA sin cuenta ngrok: localtunnel ──────────
# subprocess.run(['npm','install','-g','localtunnel'], capture_output=True)
# subprocess.Popen(['streamlit','run','app.py',
#                   '--server.port=8501','--server.headless=true'])
# time.sleep(5)
# res = subprocess.run(['npx','localtunnel','--port','8501'],
#                       capture_output=True, text=True, timeout=25)
# print('URL publica:', res.stdout.strip())


🚀  APP EN VIVO:
    https://ooze-convent-exes.ngrok-free.dev
Comparte este enlace en el stand.
Para detener: ngrok.kill()
